# AgroChain CV Dataset Preparation

This notebook documents the dataset preparation pipeline for the computer vision modules used in **AgroChain**.

It covers two image-classification datasets:

1. **Crop disease detection dataset**
2. **Soil image classification dataset**

The goal is to clean, normalize, de-duplicate, balance, and split image datasets into training and validation folders for downstream model training.

This notebook focuses only on **dataset preparation**. Model training, model evaluation, backend integration, and paper/poster figure generation are handled separately.


## Notebook Scope

This notebook includes:

- Crop disease dataset preparation
- Soil image dataset preparation
- Dataset restructuring
- Label normalization
- Duplicate removal using file hashing
- Class balancing where needed
- Train/validation split generation
- Export of final dataset folders and class labels

This notebook does not include:

- Model training
- Model evaluation
- Backend API integration
- Poster/paper figure generation
- Kaggle API credential setup



## Datasets download

The individual datasets required for this notebook can be downloaded from the :

- https://drive.google.com/drive/folders/1pEJS_vda_UMNyxYjguE132P2RSQKApk3?usp=drive_link




## 1. Environment Setup

Install only the libraries required for dataset preparation.


In [ ]:
!pip install -q tqdm pandas pillow scikit-learn

## 2. Imports

All imports are kept in one place to avoid repeated setup cells.


In [ ]:
import os
import json
import uuid
import shutil
import hashlib
import random
import zipfile
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image
from tqdm import tqdm


## 3. Global Path Configuration

Change `BASE_DIR` depending on where the datasets are stored.

For Google Colab, mount Google Drive first if your files are stored there.


In [ ]:
# Optional for Google Colab users:
# from google.colab import drive
# drive.mount('/content/drive')

# Colab / Google Drive example:
BASE_DIR = Path("/content/drive/MyDrive/Agrochain/Computer vision")

# Kaggle example:
# BASE_DIR = Path("/kaggle/input/agrochain-vision-datasets")

# Local example:
# BASE_DIR = Path("./agrochain-vision-datasets")

DISEASE_DIR = BASE_DIR / "Disease CV"
SOIL_DIR = BASE_DIR / "Soil CV"

DISEASE_DATA_DIR = DISEASE_DIR / "datasets"
DISEASE_OUTPUT_DIR = DISEASE_DIR / "outputs"

SOIL_DATA_DIR = SOIL_DIR / "datasets"
SOIL_OUTPUT_DIR = SOIL_DIR / "outputs"

for path in [DISEASE_DATA_DIR, DISEASE_OUTPUT_DIR, SOIL_DATA_DIR, SOIL_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Disease data directory:", DISEASE_DATA_DIR)
print("Soil data directory:", SOIL_DATA_DIR)


## 4. Helper Functions

These helper functions are reused by both the disease and soil dataset pipelines.


In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")


def normalize_label(label: str) -> str:
    """Normalize folder/class names into consistent machine-readable labels."""
    label = str(label).strip().lower()
    label = label.replace("___", "_")
    label = label.replace(" ", "_")
    label = label.replace("-", "_")
    label = label.replace(",", "")
    label = label.replace("(", "")
    label = label.replace(")", "")
    while "__" in label:
        label = label.replace("__", "_")
    return label.strip("_")


def reset_directory(path: Path):
    """Delete and recreate a directory."""
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def ensure_directory(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def file_md5(path: Path) -> str:
    """Generate MD5 hash for exact duplicate detection."""
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def safe_copy_image(src: Path, dst_dir: Path, prefix: str):
    """Copy an image with a unique name while preserving the file extension."""
    ensure_directory(dst_dir)
    ext = src.suffix.lower()
    dst_name = f"{prefix}_{uuid.uuid4().hex[:10]}{ext}"
    dst = dst_dir / dst_name
    shutil.copy2(src, dst)
    return dst


def count_images_by_class(root_dir: Path) -> pd.DataFrame:
    """Return image counts for each class folder under root_dir."""
    rows = []
    if not root_dir.exists():
        return pd.DataFrame(columns=["class", "images"])

    for class_dir in sorted(root_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        count = sum(1 for f in class_dir.iterdir() if f.is_file() and is_image_file(f))
        rows.append({"class": class_dir.name, "images": count})

    return pd.DataFrame(rows)


def find_class_dirs(root_dir: Path):
    """Find directories that directly contain image files."""
    class_dirs = []
    for current_root, dirs, files in os.walk(root_dir):
        current_root = Path(current_root)
        image_count = sum(1 for f in files if Path(f).suffix.lower() in IMAGE_EXTENSIONS)
        if image_count > 0:
            class_dirs.append((normalize_label(current_root.name), current_root, image_count))
    return class_dirs


# Part A: Crop Disease Dataset Preparation

This part prepares the crop disease image dataset used for AgroChain's disease detection module.


## 5. Disease Dataset Overview

The crop disease dataset was created by combining multiple public crop disease image datasets into a unified disease-level taxonomy.

The source datasets use different folder structures and naming conventions, so the main preparation steps are:

1. Restructure each dataset into disease-level folders
2. Normalize disease labels
3. Audit image counts and class overlap
4. Merge datasets while removing duplicate images
5. Create train/validation splits
6. Export class labels for model training


## 6. Disease Source Dataset Paths

Expected source datasets:

- `PlantVillage-Dataset/raw/color/`
- `PlantDoc-Dataset/train/` and `PlantDoc-Dataset/test/`
- `rice_leaf_raw/rice_leaf_diseases/`
- `multicrop_raw/`

If these folders are not present, download or attach the datasets manually before running the relevant section.(Refer above mentioned Link)




In [ ]:
PLANTVILLAGE_SOURCE = DISEASE_DATA_DIR / "PlantVillage-Dataset" / "raw" / "color"
PLANTDOC_TRAIN_SOURCE = DISEASE_DATA_DIR / "PlantDoc-Dataset" / "train"
PLANTDOC_TEST_SOURCE = DISEASE_DATA_DIR / "PlantDoc-Dataset" / "test"
RICE_SOURCE = DISEASE_DATA_DIR / "rice_leaf_raw" / "rice_leaf_diseases"
MULTICROP_SOURCE = DISEASE_DATA_DIR / "multicrop_raw"

PLANTVILLAGE_CLEAN = DISEASE_OUTPUT_DIR / "plantvillage_disease"
PLANTDOC_CLEAN = DISEASE_OUTPUT_DIR / "plantdoc_disease"
RICE_CLEAN = DISEASE_OUTPUT_DIR / "rice_leaf_disease"
MULTICROP_CLEAN = DISEASE_OUTPUT_DIR / "multicrop_disease"

DISEASE_MERGED_DIR = DISEASE_OUTPUT_DIR / "agrochain_dataset_merged"
DISEASE_FINAL_DIR = DISEASE_OUTPUT_DIR / "agrochain_dataset_final"
DISEASE_TRAIN_DIR = DISEASE_FINAL_DIR / "train"
DISEASE_VAL_DIR = DISEASE_FINAL_DIR / "val"

print("Configured disease dataset paths.")


## 8. PlantVillage Dataset Processing

PlantVillage stores images in crop-specific folders such as `Tomato___Early_blight`.

For AgroChain, these are converted into disease-level folders such as `early_blight` while preserving crop information in filenames.


In [ ]:
def process_plantvillage(source_dir: Path, output_dir: Path):
    if not source_dir.exists():
        print("PlantVillage source not found:", source_dir)
        return

    reset_directory(output_dir)
    disease_classes = set()
    total_images = 0

    for folder in sorted(source_dir.iterdir()):
        if not folder.is_dir() or "___" not in folder.name:
            continue

        crop, disease = folder.name.split("___", 1)
        disease = normalize_label(disease)
        disease_classes.add(disease)

        disease_dir = output_dir / disease
        ensure_directory(disease_dir)

        for img_path in tqdm(list(folder.iterdir()), desc=disease):
            if not img_path.is_file() or not is_image_file(img_path):
                continue
            dst_name = f"{normalize_label(crop)}_{img_path.name.strip()}"
            shutil.copy2(img_path, disease_dir / dst_name)
            total_images += 1

    print("PlantVillage restructuring complete.")
    print("Total images:", total_images)
    print("Disease classes:", len(disease_classes))


process_plantvillage(PLANTVILLAGE_SOURCE, PLANTVILLAGE_CLEAN)


## 9. PlantDoc Dataset Processing

PlantDoc uses different folder naming conventions compared to PlantVillage. This section normalizes PlantDoc labels and merges both train and test folders into a common disease-level structure.


In [ ]:
def process_plantdoc(train_dir: Path, test_dir: Path, output_dir: Path):
    sources = [train_dir, test_dir]
    if not any(path.exists() for path in sources):
        print("PlantDoc source folders not found.")
        return

    reset_directory(output_dir)
    disease_classes = set()
    total_images = 0

    for source in sources:
        if not source.exists():
            print("Skipping missing PlantDoc source:", source)
            continue

        for folder in sorted(source.iterdir()):
            if not folder.is_dir():
                continue

            disease = normalize_label(folder.name)
            disease_classes.add(disease)
            disease_dir = output_dir / disease
            ensure_directory(disease_dir)

            for img_path in tqdm(list(folder.iterdir()), desc=disease):
                if not img_path.is_file() or not is_image_file(img_path):
                    continue
                dst_name = f"{disease}_{img_path.name.strip()}"
                shutil.copy2(img_path, disease_dir / dst_name)
                total_images += 1

    print("PlantDoc restructuring complete.")
    print("Total images:", total_images)
    print("Disease classes:", len(disease_classes))


process_plantdoc(PLANTDOC_TRAIN_SOURCE, PLANTDOC_TEST_SOURCE, PLANTDOC_CLEAN)


## 10. Rice Leaf Disease Dataset Processing

The rice leaf disease dataset is processed separately because it contains rice-specific disease folders.


In [ ]:
def process_rice_dataset(source_dir: Path, output_dir: Path):
    if not source_dir.exists():
        print("Rice disease source not found:", source_dir)
        return

    reset_directory(output_dir)
    disease_classes = set()
    total_images = 0

    for folder in sorted(source_dir.iterdir()):
        if not folder.is_dir():
            continue

        disease = normalize_label(folder.name)
        disease_classes.add(disease)
        disease_dir = output_dir / disease
        ensure_directory(disease_dir)

        for img_path in tqdm(list(folder.iterdir()), desc=disease):
            if not img_path.is_file() or not is_image_file(img_path):
                continue
            dst_name = f"rice_{img_path.name.strip()}"
            shutil.copy2(img_path, disease_dir / dst_name)
            total_images += 1

    print("Rice dataset restructuring complete.")
    print("Total images:", total_images)
    print("Disease classes:", len(disease_classes))


process_rice_dataset(RICE_SOURCE, RICE_CLEAN)


## 11. Multi-Crop Disease Dataset Processing

The multi-crop disease dataset contains nested crop and disease folders. This section flattens the folder structure into disease-level categories.


In [ ]:
def process_multicrop_dataset(source_dir: Path, output_dir: Path):
    if not source_dir.exists():
        print("Multi-crop disease source not found:", source_dir)
        return

    reset_directory(output_dir)
    disease_classes = set()
    total_images = 0

    for crop_group in sorted(source_dir.iterdir()):
        if not crop_group.is_dir():
            continue

        for disease_folder in sorted(crop_group.iterdir()):
            if not disease_folder.is_dir():
                continue

            disease_name = normalize_label(disease_folder.name)
            disease_classes.add(disease_name)
            target_dir = output_dir / disease_name
            ensure_directory(target_dir)

            # Some folders contain another nested directory; others contain images directly.
            for item in disease_folder.iterdir():
                if item.is_dir():
                    for img_path in tqdm(list(item.iterdir()), desc=disease_name):
                        if not img_path.is_file() or not is_image_file(img_path):
                            continue
                        dst_name = f"{normalize_label(item.name)}_{img_path.name.strip()}"
                        shutil.copy2(img_path, target_dir / dst_name)
                        total_images += 1
                elif item.is_file() and is_image_file(item):
                    dst_name = f"{disease_name}_{item.name.strip()}"
                    shutil.copy2(item, target_dir / dst_name)
                    total_images += 1

    print("Multi-crop disease dataset restructuring complete.")
    print("Total images:", total_images)
    print("Disease classes:", len(disease_classes))


process_multicrop_dataset(MULTICROP_SOURCE, MULTICROP_CLEAN)


## 12. Disease Dataset Audit

Before merging, audit each cleaned source dataset to identify class counts, overlapping labels, and imbalance.


In [ ]:
DISEASE_CLEAN_DATASETS = {
    "plantvillage": PLANTVILLAGE_CLEAN,
    "plantdoc": PLANTDOC_CLEAN,
    "rice": RICE_CLEAN,
    "multicrop": MULTICROP_CLEAN,
}

stats = []

for dataset_name, dataset_path in DISEASE_CLEAN_DATASETS.items():
    if not dataset_path.exists():
        print("Skipping missing dataset:", dataset_name, dataset_path)
        continue

    class_counts = count_images_by_class(dataset_path)
    class_counts["dataset"] = dataset_name
    stats.append(class_counts)

if stats:
    disease_audit_df = pd.concat(stats, ignore_index=True)[["dataset", "class", "images"]]
    display(disease_audit_df.sort_values("images", ascending=False).head(30))

    print("Total datasets:", disease_audit_df["dataset"].nunique())
    print("Total unique labels before mapping:", disease_audit_df["class"].nunique())
    print("Total images before merge:", disease_audit_df["images"].sum())

    overlapping = disease_audit_df.groupby("class")["dataset"].nunique()
    overlapping = overlapping[overlapping > 1].sort_values(ascending=False)
    print("Labels appearing in multiple datasets:")
    print(overlapping.head(30))

    audit_path = DISEASE_OUTPUT_DIR / "disease_dataset_analysis.csv"
    disease_audit_df.to_csv(audit_path, index=False)
    print("
Saved audit CSV:", audit_path)
else:
    print("No cleaned disease datasets found for audit.")


## 13. Unified Disease Label Mapping

The source datasets contain many crop-specific labels. For AgroChain, these labels are mapped into a smaller disease-level taxonomy to support generalized disease classification.

The final `classes.json` is generated automatically from the final train folders after the split, so it always matches the exported dataset.


In [ ]:
DISEASE_LABEL_MAP = {
    # healthy
    "healthy": "healthy",
    "healthy_maize": "healthy",
    "healthy_wheat": "healthy",
    "sugarcane_healthy": "healthy",
    "healthy_cotton": "healthy",

    # blights
    "early_blight": "early_blight",
    "potato_leaf_early_blight": "early_blight",
    "tomato_early_blight_leaf": "early_blight",
    "late_blight": "late_blight",
    "potato_leaf_late_blight": "late_blight",
    "tomato_leaf_late_blight": "late_blight",

    # rust
    "common_rust": "rust",
    "common_rust_": "rust",
    "corn_rust_leaf": "rust",
    "wheat_brown_leaf_rust": "rust",
    "wheat_black_rust": "rust",
    "wheat_yellow_rust": "rust",
    "yellow_rust_sugarcane": "rust",
    "redrust_sugarcane": "rust",

    # leaf spot
    "leaf_spot": "leaf_spot",
    "bell_pepper_leaf_spot": "leaf_spot",
    "gray_leaf_spot": "leaf_spot",
    "cercospora_leaf_spot_gray_leaf_spot": "leaf_spot",
    "septoria_leaf_spot": "leaf_spot",
    "tomato_septoria_leaf_spot": "leaf_spot",

    # mildew
    "powdery_mildew": "powdery_mildew",
    "squash_powdery_mildew_leaf": "powdery_mildew",
    "wheat_powdery_mildew": "powdery_mildew",

    # scab
    "apple_scab": "scab",
    "apple_scab_leaf": "scab",
    "wheat_scab": "scab",

    # bacterial
    "bacterial_spot": "bacterial",
    "bacterial_leaf_blight": "bacterial",
    "becterial_blight_in_rice": "bacterial",
    "bacterial_blight_in_cotton": "bacterial",
    "tomato_leaf_bacterial_spot": "bacterial",

    # virus
    "tomato_mosaic_virus": "virus",
    "tomato_leaf_mosaic_virus": "virus",
    "tomato_yellow_leaf_curl_virus": "virus",
    "tomato_leaf_yellow_virus": "virus",
    "tungro": "virus",
    "mosaic_sugarcane": "virus",

    # mold
    "leaf_mold": "leaf_mold",
    "tomato_mold_leaf": "leaf_mold",

    # rot
    "black_rot": "rot",
    "grape_leaf_black_rot": "rot",
    "maize_ear_rot": "rot",
    "redrot_sugarcane": "rot",
    "bollrot_on_cotton": "rot",

    # smut and wilt
    "leaf_smut": "smut",
    "flag_smut": "smut",
    "wilt": "wilt",

    # mites
    "spider_mites_two_spotted_spider_mite": "mites",
    "spider_mites_two-spotted_spider_mite": "mites",
    "tomato_two_spotted_spider_mites_leaf": "mites",
    "wheat_mite": "mites",

    # pests
    "cotton_aphid": "pest",
    "wheat_aphid": "pest",
    "army_worm": "pest",
    "maize_fall_armyworm": "pest",
    "bollworm_on_cotton": "pest",
    "pink_bollworm_in_cotton": "pest",
    "american_bollworm_on_cotton": "pest",
    "maize_stem_borer": "pest",
    "cotton_whitefly": "pest",
}

print("Disease label mappings:", len(DISEASE_LABEL_MAP))
print("Target classes:", sorted(set(DISEASE_LABEL_MAP.values())))


## 14. Disease Dataset Duplicate Removal, Merge, and Train/Validation Split

Since multiple public datasets may contain overlapping images, duplicate removal is performed using file hashes before creating the final dataset.

The final dataset is split class-wise into training and validation folders while preserving the same class structure.


In [ ]:
def merge_and_split_disease_dataset(
    datasets: dict,
    label_map: dict,
    merged_dir: Path,
    final_dir: Path,
    train_ratio: float = 0.80,
    max_healthy: int = 7000,
    seed: int = 42,
):
    random.seed(seed)

    train_dir = final_dir / "train"
    val_dir = final_dir / "val"

    reset_directory(merged_dir)
    reset_directory(final_dir)
    ensure_directory(train_dir)
    ensure_directory(val_dir)

    hashes = set()
    merged = 0
    duplicates = 0
    skipped_unmapped = Counter()

    print("--- MERGING DISEASE DATASETS ---")

    for dataset_name, dataset_path in datasets.items():
        if not dataset_path.exists():
            print("Skipping missing dataset:", dataset_name, dataset_path)
            continue

        print("Processing:", dataset_name)

        for class_dir in sorted(dataset_path.iterdir()):
            if not class_dir.is_dir():
                continue

            source_label = normalize_label(class_dir.name)
            if source_label not in label_map:
                skipped_unmapped[source_label] += 1
                continue

            target_class = label_map[source_label]
            target_dir = merged_dir / target_class
            ensure_directory(target_dir)

            for img_path in tqdm(list(class_dir.iterdir()), desc=source_label):
                if not img_path.is_file() or not is_image_file(img_path):
                    continue

                try:
                    h = file_md5(img_path)
                    if h in hashes:
                        duplicates += 1
                        continue

                    hashes.add(h)
                    safe_copy_image(img_path, target_dir, dataset_name)
                    merged += 1
                except Exception:
                    continue

    print("Merged images:", merged)
    print("Duplicates removed:", duplicates)
    print("Unmapped labels skipped:", len(skipped_unmapped))

    if skipped_unmapped:
        skipped_df = pd.DataFrame(
            [{"label": label, "count": count} for label, count in skipped_unmapped.items()]
        ).sort_values("label")
        skipped_path = DISEASE_OUTPUT_DIR / "unmapped_disease_labels.csv"
        skipped_df.to_csv(skipped_path, index=False)
        print("Saved unmapped-label report:", skipped_path)

    print("--- TRAIN/VALIDATION SPLIT ---")

    for class_dir in sorted(merged_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [p for p in class_dir.iterdir() if p.is_file() and is_image_file(p)]

        # Healthy images can dominate merged public datasets, so cap them if needed.
        if class_dir.name == "healthy" and max_healthy is not None:
            images = random.sample(images, min(len(images), max_healthy))

        random.shuffle(images)
        split_idx = int(train_ratio * len(images))

        train_images = images[:split_idx]
        val_images = images[split_idx:]

        train_class_dir = train_dir / class_dir.name
        val_class_dir = val_dir / class_dir.name
        ensure_directory(train_class_dir)
        ensure_directory(val_class_dir)

        for img_path in tqdm(train_images, desc=f"train {class_dir.name}"):
            shutil.copy2(img_path, train_class_dir / img_path.name)

        for img_path in tqdm(val_images, desc=f"val {class_dir.name}"):
            shutil.copy2(img_path, val_class_dir / img_path.name)

    classes = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    classes_path = final_dir / "classes.json"
    with open(classes_path, "w") as f:
        json.dump(classes, f, indent=2)

    print("
Disease dataset pipeline complete.")
    print("Classes:", classes)
    print("Saved classes.json:", classes_path)

    return {
        "merged_images": merged,
        "duplicates_removed": duplicates,
        "classes": classes,
        "classes_path": classes_path,
    }


disease_result = merge_and_split_disease_dataset(
    datasets=DISEASE_CLEAN_DATASETS,
    label_map=DISEASE_LABEL_MAP,
    merged_dir=DISEASE_MERGED_DIR,
    final_dir=DISEASE_FINAL_DIR,
    train_ratio=0.80,
    max_healthy=7000,
    seed=42,
)


## 15. Disease Dataset Exported Artifacts

The final outputs of the disease pipeline are:

- `outputs/agrochain_dataset_final/train/`
- `outputs/agrochain_dataset_final/val/`
- `outputs/agrochain_dataset_final/classes.json`

These artifacts are used for training the AgroChain crop disease detection model.


In [ ]:
train_counts = count_images_by_class(DISEASE_TRAIN_DIR)
val_counts = count_images_by_class(DISEASE_VAL_DIR)

if not train_counts.empty:
    summary = train_counts.merge(val_counts, on="class", how="outer", suffixes=("_train", "_val")).fillna(0)
    summary["total"] = summary["images_train"] + summary["images_val"]
    display(summary.sort_values("total", ascending=False))
else:
    print("Disease final train folder not found or empty:", DISEASE_TRAIN_DIR)


# Part B: Soil Image Dataset Preparation

This part prepares the soil image dataset used for AgroChain's soil classification module.


## 16. Soil Dataset Overview

The soil dataset preparation pipeline combines soil image folders, normalizes labels into common soil categories, removes duplicate images, balances classes, and creates train/validation splits.


## 17. Soil Dataset Extraction

The source soil image datasets were stored as zip files and extracted into a common dataset directory.

This section extracts each zip file only if it has not already been extracted.


In [ ]:
def extract_soil_zip_datasets(data_dir: Path):
    if not data_dir.exists():
        print("Soil data directory not found:", data_dir)
        return

    for zip_path in sorted(data_dir.glob("*.zip")):
        extract_path = data_dir / zip_path.stem

        if extract_path.exists():
            print("Already extracted:", zip_path.name)
            continue

        print("Extracting:", zip_path.name)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)

    print("Soil dataset extraction complete.")


extract_soil_zip_datasets(SOIL_DATA_DIR)


## 18. Soil Dataset Audit

This section scans the extracted soil image folders and counts available images per detected class.


In [ ]:
def audit_soil_datasets(data_dir: Path, output_dir: Path):
    dataset_stats = []
    format_counter = Counter()
    corrupted = 0

    if not data_dir.exists():
        print("Soil data directory not found:", data_dir)
        return pd.DataFrame()

    for dataset_path in sorted(data_dir.iterdir()):
        if not dataset_path.is_dir():
            continue

        print("============================")
        print("DATASET:", dataset_path.name)

        class_dirs = find_class_dirs(dataset_path)
        if not class_dirs:
            print("No image classes detected")
            continue

        total_images = 0

        for class_name, class_path, count in class_dirs:
            print(f"{class_name:25} {count}")
            dataset_stats.append({
                "dataset": dataset_path.name,
                "class": class_name,
                "images": count,
            })
            total_images += count

            for file_name in os.listdir(class_path):
                file_path = class_path / file_name
                ext = file_path.suffix.lower()
                if ext in IMAGE_EXTENSIONS:
                    format_counter[ext] += 1
                    try:
                        Image.open(file_path).verify()
                    except Exception:
                        corrupted += 1

        print("Total images:", total_images)

    df = pd.DataFrame(dataset_stats)

    if not df.empty:
        print("============================")
        print("SOIL DATASET SUMMARY")
        print("============================")
        print("Total datasets:", df["dataset"].nunique())
        print("Total detected classes:", df["class"].nunique())
        print("Total images:", df["images"].sum())
        print("Image formats:", format_counter)
        print("Corrupted images:", corrupted)

        audit_path = output_dir / "soil_dataset_analysis.csv"
        df.to_csv(audit_path, index=False)
        print("Saved soil audit CSV:", audit_path)
        display(df.sort_values("images", ascending=False).head(30))
    else:
        print("No soil image classes found.")

    return df


soil_audit_df = audit_soil_datasets(SOIL_DATA_DIR, SOIL_OUTPUT_DIR)


## 19. Unified Soil Label Mapping

Different soil datasets use different names for similar soil types. These labels are normalized into four common categories:

- `clay`
- `sandy`
- `loam`
- `silt`


In [ ]:
SOIL_LABEL_MAP = {
    # clay
    "clay": "clay",
    "clay_soil": "clay",
    "clayey_soils": "clay",
    "black_soil": "clay",
    "black": "clay",
    "laterite_soil": "clay",
    "laterite": "clay",

    # sandy
    "red_soil": "sandy",
    "red": "sandy",
    "sandy": "sandy",
    "sandy_soil": "sandy",
    "sandy_loam": "sandy",
    "yellow": "sandy",
    "yellow_soil": "sandy",
    "cinder": "sandy",
    "cinder_soil": "sandy",

    # loam
    "loamy": "loam",
    "loamy_soil": "loam",
    "alluvial": "loam",
    "alluvial_soil": "loam",
    "sandy_loam": "loam",

    # silt
    "peat": "silt",
    "peat_soil": "silt",
    "mountain_soil": "silt",
    "arid_soil": "silt",
}

print("Soil label mappings:", len(SOIL_LABEL_MAP))
print("Target soil classes:", sorted(set(SOIL_LABEL_MAP.values())))


## 20. Soil Dataset Duplicate Removal, Class Balancing, and Train/Validation Split

Duplicate soil images are removed using file hashing to avoid repeated samples after merging datasets.

Class balancing is applied to avoid one soil category dominating the training dataset.


In [ ]:
def merge_and_split_soil_dataset(
    data_dir: Path,
    label_map: dict,
    output_dir: Path,
    train_ratio: float = 0.80,
    max_per_class: int = 5000,
    seed: int = 42,
):
    random.seed(seed)

    merged_dir = output_dir / "soil_dataset_merged"
    final_dir = output_dir / "agrochain_soil_dataset_final"
    train_dir = final_dir / "train"
    val_dir = final_dir / "val"

    reset_directory(merged_dir)
    reset_directory(final_dir)
    ensure_directory(train_dir)
    ensure_directory(val_dir)

    hashes = set()
    merged = 0
    duplicates = 0
    skipped_unmapped = Counter()

    print("--- MERGING SOIL DATASETS ---")

    if not data_dir.exists():
        print("Soil data directory not found:", data_dir)
        return None

    for dataset_path in sorted(data_dir.iterdir()):
        if not dataset_path.is_dir():
            continue

        print("Processing:", dataset_path.name)
        class_dirs = find_class_dirs(dataset_path)

        for source_label, class_path, _ in class_dirs:
            source_label = normalize_label(source_label)

            if source_label not in label_map:
                skipped_unmapped[source_label] += 1
                continue

            target_class = label_map[source_label]
            target_dir = merged_dir / target_class
            ensure_directory(target_dir)

            for img_path in tqdm(list(class_path.iterdir()), desc=source_label):
                if not img_path.is_file() or not is_image_file(img_path):
                    continue

                try:
                    h = file_md5(img_path)
                    if h in hashes:
                        duplicates += 1
                        continue

                    hashes.add(h)
                    safe_copy_image(img_path, target_dir, dataset_path.name)
                    merged += 1
                except Exception:
                    continue

    print("Merged soil images:", merged)
    print("Duplicates removed:", duplicates)
    print("Unmapped soil labels skipped:", len(skipped_unmapped))

    if skipped_unmapped:
        skipped_df = pd.DataFrame(
            [{"label": label, "count": count} for label, count in skipped_unmapped.items()]
        ).sort_values("label")
        skipped_path = output_dir / "unmapped_soil_labels.csv"
        skipped_df.to_csv(skipped_path, index=False)
        print("Saved unmapped soil-label report:", skipped_path)

    print("--- BALANCING SOIL DATASET ---")

    for class_dir in sorted(merged_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [p for p in class_dir.iterdir() if p.is_file() and is_image_file(p)]

        if max_per_class is not None and len(images) > max_per_class:
            remove_images = random.sample(images, len(images) - max_per_class)
            for img_path in remove_images:
                img_path.unlink()

        print(class_dir.name, len([p for p in class_dir.iterdir() if p.is_file() and is_image_file(p)]))

    print("--- TRAIN/VALIDATION SPLIT ---")

    for class_dir in sorted(merged_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [p for p in class_dir.iterdir() if p.is_file() and is_image_file(p)]
        random.shuffle(images)
        split_idx = int(train_ratio * len(images))

        train_images = images[:split_idx]
        val_images = images[split_idx:]

        train_class_dir = train_dir / class_dir.name
        val_class_dir = val_dir / class_dir.name
        ensure_directory(train_class_dir)
        ensure_directory(val_class_dir)

        for img_path in train_images:
            shutil.copy2(img_path, train_class_dir / img_path.name)

        for img_path in val_images:
            shutil.copy2(img_path, val_class_dir / img_path.name)

    classes = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    classes_path = final_dir / "soil_classes.json"
    with open(classes_path, "w") as f:
        json.dump(classes, f, indent=2)

    print("Soil dataset pipeline complete.")
    print("Classes:", classes)
    print("Saved soil_classes.json:", classes_path)

    return {
        "merged_images": merged,
        "duplicates_removed": duplicates,
        "classes": classes,
        "classes_path": classes_path,
    }


soil_result = merge_and_split_soil_dataset(
    data_dir=SOIL_DATA_DIR,
    label_map=SOIL_LABEL_MAP,
    output_dir=SOIL_OUTPUT_DIR,
    train_ratio=0.80,
    max_per_class=5000,
    seed=42,
)


## 21. Soil Dataset Exported Artifacts

The final outputs of the soil pipeline are:

- `outputs/agrochain_soil_dataset_final/train/`
- `outputs/agrochain_soil_dataset_final/val/`
- `outputs/agrochain_soil_dataset_final/soil_classes.json`

These artifacts are used for training the AgroChain soil image classification model.


In [ ]:
SOIL_FINAL_DIR = SOIL_OUTPUT_DIR / "agrochain_soil_dataset_final"
SOIL_TRAIN_DIR = SOIL_FINAL_DIR / "train"
SOIL_VAL_DIR = SOIL_FINAL_DIR / "val"

train_counts = count_images_by_class(SOIL_TRAIN_DIR)
val_counts = count_images_by_class(SOIL_VAL_DIR)

if not train_counts.empty:
    summary = train_counts.merge(val_counts, on="class", how="outer", suffixes=("_train", "_val")).fillna(0)
    summary["total"] = summary["images_train"] + summary["images_val"]
    display(summary.sort_values("total", ascending=False))
else:
    print("Soil final train folder not found or empty:", SOIL_TRAIN_DIR)


# Final Notes: AgroChain Integration

The final outputs of this notebook are used by the AgroChain computer vision pipeline.

## Disease Dataset

- `agrochain_dataset_final/train/`
- `agrochain_dataset_final/val/`
- `classes.json`

## Soil Dataset

- `agrochain_soil_dataset_final/train/`
- `agrochain_soil_dataset_final/val/`
- `soil_classes.json`

These datasets support AgroChain's image-based modules and are used alongside the yield prediction, soil/climate, RAG advisory, and blockchain verification components.


